In [ ]:
# ===========================================================
# 1️⃣ Imports
# ===========================================================
import pandas as pd
import numpy as np
import json
from tqdm import tqdm

import torch
from torch.utils.data import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split


from main import extract_data_from_features, extract_method, stop_steps

In [ ]:
# Création d'un fichier json pour garder les valeurs
import json

with open("results.json", "w") as f:
    method = extract_method()
    f[method] = {}
    json.dump(f)

In [ ]:
# =========================================================== 
# 1️⃣ Fonctions de chargement JSONL
# ===========================================================
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return pd.DataFrame(rows)

In [ ]:
# ===========================================================
# 2 Construction du DataFrame avec features
# Injecte les features dans le texte du tweet
# ===========================================================
def prepare_dataframe(df, is_train=True):
    final_texts = []

    for _, row in df.iterrows():
        txt = row.get("text", "")
        user_desc = extract_data_from_features(row)

        # Fusion optimisée
        fused = f"{txt}\n\n[PROFILE] {user_desc}"
        final_texts.append(fused)

    out = pd.DataFrame()
    out["text"] = final_texts

    if is_train and "label" in df:
        out["label"] = df["label"]

    # Garder ID si présent (pour submission)
    if "challenge_id" in df:
        out["ID"] = df["challenge_id"]
    elif "id_str" in df:
        out["ID"] = df["id_str"]

    return out

In [ ]:
# ===========================================================
# 5️⃣ Charger les datasets
# ===========================================================
train_raw = load_jsonl("../../../data/train.jsonl")
test_raw = load_jsonl("../../../data/kaggle_test.jsonl")

train_df = prepare_dataframe(train_raw, is_train=True)
test_df = prepare_dataframe(test_raw, is_train=False)

# ===========================================================

In [ ]:
# ===========================================================
# 6️⃣ Train / Validation split
# ===========================================================
train_split, val_split = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df["label"]
)

# ===========================================================
# 7️⃣ Dataset PyTorch
# ===========================================================
MODEL_NAME = "vinai/bertweet-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

class TweetDataset(Dataset):
    def __init__(self, df):
        self.texts = df["text"].tolist()
        self.labels = df["label"].tolist() if "label" in df else None

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=128,
            padding="max_length",
            return_tensors="pt"
        )
        item = {key: val.squeeze(0) for key, val in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


train_dataset = TweetDataset(train_split)
val_dataset   = TweetDataset(val_split)
test_dataset  = TweetDataset(test_df)

# ===========================================================

In [ ]:
from transformers import TrainerCallback
import time

class MyCallback(TrainerCallback):
    def __init__(self, trainer, test_dataset, test_df, acc_threshold=0.82):
        self.trainer = trainer
        self.test_dataset = test_dataset
        self.test_df = test_df
        self.acc_threshold = acc_threshold

    def on_evaluate(self, args, state, control, **kwargs):
        print(f"Step {state.global_step} – evaluation terminée.")

        # Récupération des métriques (eval_loss, eval_accuracy, etc.)
        metrics = kwargs.get("metrics", {})
        eval_acc = metrics.get("eval_accuracy", None)

        if eval_acc is not None:
            print(f"📊 Accuracy de validation = {eval_acc:.4f}")

            with open("results.json", "r") as f:
                data = json.load(f)
            method = extract_method()
            if method not in data:
                data[method] = {}
            data[method][str(state.global_step)] = eval_acc
        else:
            print("⚠️ Impossible de récupérer eval_accuracy.")
            return

        # Pour arrêter le training à un certain nombre de steps
        if state.global_step >= stop_steps:
            raise SystemExit("Done")
        
        # Condition : steps == multiple de 2000 + accuracy > threshold
        if state.global_step % 2000 == 0 and eval_acc >= self.acc_threshold:
            print(f"🚀 Accuracy > {self.acc_threshold} → Génération du fichier de soumission...")

            start = time.time()
            preds = self.trainer.predict(self.test_dataset)
            labels = preds.predictions.argmax(axis=1)
            end = time.time()

            print(f"⏱ Prédictions test terminées en {end - start:.2f} sec.")

            submission = pd.DataFrame({
                "ID": self.test_df["ID"],
                "Prediction": labels
            })

            filename = f"submission_step_{state.global_step}_acc_{eval_acc:.4f}.csv"
            submission.to_csv(filename, index=False)

            print(f"✅ {filename} généré (accuracy={eval_acc:.4f})")

        elif state.global_step % 2000 == 0:
            print(f"❌ Accuracy {eval_acc:.4f} < {self.acc_threshold} → pas de submission.")

    def on_epoch_end(self, args, state, control, **kwargs):
        print(f"Epoch {state.epoch} terminée.")

In [ ]:
# ===========================================================
# 9️⃣ Modèle et entraînement
# ===========================================================
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=2
)

training_args = TrainingArguments(
    output_dir="./bertweet_output",
    eval_strategy="steps",
    eval_steps=500,
    logging_strategy="steps",
    logging_steps=500,
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
)

def compute_metrics(preds):
    logits, labels = preds.predictions, preds.label_ids
    acc = (logits.argmax(axis=-1) == labels).mean()
    return {"accuracy": acc}

callback = MyCallback(trainer=None, test_dataset=test_dataset, test_df=test_df)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[callback]
)

callback.trainer = trainer

trainer.train()

In [ ]:

# ===========================================================
# 0️⃣ Prédiction test + submission
# ===========================================================
preds = trainer.predict(test_dataset)
labels = preds.predictions.argmax(axis=1)

submission = pd.DataFrame({
    "ID": test_df["ID"],
    "Prediction": labels
})

submission.to_csv("submission.csv", index=False)
print("✅ submission.csv généré !")
